# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mehroze-Ali/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata

# Authenticate and connect
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
base = "hf://datasets/FlyRank/internship-warehouse"

# Aggregate the March 2026 data to get a monthly view per content piece
print("--- Aggregating Base Metrics for the Rule ---")
q_agg = f"""
SELECT
    content_hash_id,
    SUM(gsc_impressions) as total_impressions,
    SUM(gsc_clicks) as total_clicks,
    AVG(gsc_avg_position) as avg_position
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY content_hash_id
HAVING total_impressions > 100 -- Filter out complete ghost pages
"""
df = con.sql(q_agg).df()
print(f"Loaded {len(df)} content items for scoring.")


--- Aggregating Base Metrics for the Rule ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 101232 content items for scoring.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

def calculate_baseline_score(row):
    score = 0
    reasons = []

    # Calculate CTR (safeguard against division by zero)
    ctr = (row['total_clicks'] / row['total_impressions']) * 100 if row['total_impressions'] > 0 else 0

    # Rule 1: High volume, bad rank (50 points)
    if row['total_impressions'] > 1000 and row['avg_position'] > 10:
        score += 50
        reasons.append("high_volume_low_rank")

    # Rule 2: High volume, terrible CTR (30 points)
    if row['total_impressions'] > 500 and ctr < 2.0:
        score += 30
        reasons.append("low_ctr_high_imp")

    action = 'refresh_priority' if score >= 50 else ('monitor' if score > 0 else 'ignore')
    return pd.Series([score, " | ".join(reasons), action, round(ctr, 2)])

# Apply the rules
df[['baseline_score', 'reason_codes', 'action_label', 'ctr_pct']] = df.apply(calculate_baseline_score, axis=1)

# Rank the queue
df_ranked = df.sort_values(by=['baseline_score', 'total_impressions'], ascending=[False, False])

# Write the CSV to the correct directory (creating it if it doesn't exist)
os.makedirs('../../work/outputs', exist_ok=True)
csv_path = '../../work/outputs/baseline_action_score.csv'

# Fallback path if working directory is already at repo root
if not os.path.exists('../../work'):
    os.makedirs('work/outputs', exist_ok=True)
    csv_path = 'work/outputs/baseline_action_score.csv'

df_ranked.to_csv(csv_path, index=False)
print(f"Successfully wrote ranked queue to {csv_path}")


Successfully wrote ranked queue to ../../work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display the top 20 so we can actually see the metrics we are reviewing
print("--- Top 20 Candidates for Refresh ---")
display(df_ranked[['content_hash_id', 'total_impressions', 'avg_position', 'ctr_pct', 'baseline_score', 'reason_codes']].head(20))


--- Top 20 Candidates for Refresh ---


,content_hash_id,total_impressions,avg_position,ctr_pct,baseline_score,reason_codes
46231,content_e8a52cf3d5988c07,244931.0,15.008339,0.27,80,high_volume_low_rank | low_ctr_high_imp
97332,content_36e53e9c707674fc,194579.0,32.766674,0.12,80,high_volume_low_rank | low_ctr_high_imp
22851,content_82e35c4845e6c391,143907.0,22.558608,0.04,80,high_volume_low_rank | low_ctr_high_imp
96888,content_3df3f32f3fd58dea,140156.0,23.335465,0.14,80,high_volume_low_rank | low_ctr_high_imp
48285,content_66288edeb93b7c4f,137878.0,18.615742,0.57,80,high_volume_low_rank | low_ctr_high_imp
45925,content_df47d1b976106de4,131707.0,24.355625,0.12,80,high_volume_low_rank | low_ctr_high_imp
46335,content_5e1c049f62e33b11,120175.0,18.077081,0.14,80,high_volume_low_rank | low_ctr_high_imp
96822,content_bdf60c86117079be,112429.0,30.769353,0.01,80,high_volume_low_rank | low_ctr_high_imp
6994,content_661a7734f691bef5,110424.0,23.888656,0.07,80,high_volume_low_rank | low_ctr_high_imp
56779,content_cae701a83cad5e36,98572.0,23.730705,0.25,80,high_volume_low_rank | low_ctr_high_imp


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage check: prove no future dates snuck into our scoring dataframe
q_leak = f"""
SELECT MAX(report_date) as latest_date_used
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
print("--- Leakage Check ---")
display(con.sql(q_leak).df())


--- Leakage Check ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,latest_date_used
0,2026-03-31


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.